# Imports and initialization

In [1]:
import re

import logging
import os
from pathlib import Path
import pandas as pd

import spacy
from spacy import displacy
from spacy.tokens.doc import Doc

# defining paths for notebook
current_dir = Path(globals()["_dh"][0])

csv_folder = os.path.join(current_dir, r"csv")
train_csv = "train.csv"

# logging configuration
logging.basicConfig(level=logging.DEBUG)

## Reading csv

In [2]:
# We are not using title, but we'd like to, add it here
used_columns = ["id", "text", "rating"]

df = pd.read_csv(os.path.join(csv_folder, train_csv), usecols=used_columns)

text_test = df["text"][10]

df.head()

,id,text,rating
0,17578,"Por incrível que pareça, para uma bebida desti...",5
1,18658,"O readset pode até ser bom, mais tem outros fo...",1
2,28477,"Foi difícil terminar esse livro , demorou mese...",2
3,43638,"A bola é boa divertida, mas não é nem um pouco...",2
4,26130,Comprei errado! Não tenho leitor de e-books. Q...,1


In [3]:
print(df['rating'].value_counts())

rating
5    8230
1    8205
4    8198
3    8194
2    8178
Name: count, dtype: int64


# Cleaning text

In [4]:
def clean_text(text):
    text = (
        text.lower()
    )  # NOTE: talvez seja interessante deixar alguns caracteres em maiúsculo

    # remove URLs
    text = re.sub(r"http\\S+|www\\S+", " ", text)

    # keeps letters and spaces
    text = re.sub(
        r"[^a-zà-úâêîôûãõç\\s]", " ", text
    )  # NOTE: está tirando os números também

    # removes extra spaces
    text = re.sub(r"\\s+", " ", text).strip()

    return text

In [5]:
print(clean_text(text_test))

ótimo custo benefício  não tampa tão bem o sol  mas pelo valor é ok  enteguega o que propõe


# Features

In [6]:
# ADJ - ADJETIVO
# ADV - ADVERBIO
# INTJ - INTERJEIÇÃO
# NUM - NÚMERO POR EXTENSO
# PUNCT - PONTUAÇÃO (EXCLAMACAO)
# SYMBOLS - SIMBOLOS (R$)
# COMP - COMPARACAO EU ACHO (A MELHOR PIZZARIA DA CIDADE)


# 1 - Number of ADJ  
# 3 - Number of ADV
# 6 - Number of DET 
# 7 - Number of INTJ 
# 9 - Number of NUM 
# 12 - Number of PUNCT 
# 14 - Number of SYM 
# 16 - Number of COMP 
# 17 - Number of SUP 
# 18 - Number of X 

# 19 ADJ → NOUN O melhor fondue que j ́a comi!!!!
# (The best fondue I’ve ever had!!!!)
# 1

# 20 ADV → ADJ → Not (NOUN) Vergonhoso cobrarem t ̃ao caro.
# (It’s shameful they charge so expensive.)
# 1
# 21 ADJ → ADJ → Not (NOUN) Uma explos ̃ao de sabores regionais inigual ́avel!
# (An unequaled explosion of regional
# flavors!)

# 1

# 22 NOUN → ADJ → Not (NOUN) Pre ̧cos bem salgados por sinal.
# (Very salty prices by the way.)
# 1

# 23 ADV → VERB + {PCP | GER | PS | IMPF} Fomos muito bem atendidos!
# (We were very well served!)
# 24-31
#
# tabela 4 ignora
#
# 24
#
# 46
#
# tabela 6 twitter

## Part-Of-Speech (POS) features

Don't forget to `python -m spacy download pt_core_news_sm`
See https://universaldependencies.org/u/pos/ for information on Tags

In [7]:
nlp = spacy.load("pt_core_news_lg")

In [8]:
doc_test = nlp(text_test)

In [9]:
print(doc_test)

Ótimo custo benefício, não tampa tão bem o sol, mas pelo valor é ok. Enteguega o que propõe.


In [10]:
# Feature 1
def number_of_adj(doc: Doc):
    logging.debug("[number_of_adj]")

    adj_num = 0
    for token in doc:
        if token.pos_ == "ADJ":
            adj_num += 1

            logging.debug(token.text + "\t|" + token.pos_)
    return adj_num


print(doc_test, "\nNumber of adj:", number_of_adj(doc_test))

DEBUG:root:[number_of_adj]
DEBUG:root:benefício	|ADJ


Ótimo custo benefício, não tampa tão bem o sol, mas pelo valor é ok. Enteguega o que propõe. 
Number of adj: 1


In [11]:
def number_of_adv(doc: Doc):
    adv_num = 0

    for token in doc:
        if token.pos_ == "ADJ":
            adv_num += 1
            logging.debug(token.text + "\t|" + token.pos_)

    return adv_num


print(doc_test, "\nNumber of adv:", number_of_adv(doc_test))

DEBUG:root:benefício	|ADJ


Ótimo custo benefício, não tampa tão bem o sol, mas pelo valor é ok. Enteguega o que propõe. 
Number of adv: 1


### spaCy testing

In [12]:
doc = nlp(text_test)
for token in doc:
    print(token.text, "\t\t|", token.pos_, token.dep_)

Ótimo 		| DET det
custo 		| NOUN nsubj
benefício 		| ADJ amod
, 		| PUNCT punct
não 		| ADV advmod
tampa 		| VERB parataxis
tão 		| ADV advmod
bem 		| ADV advmod
o 		| DET det
sol 		| NOUN obj
, 		| PUNCT punct
mas 		| CCONJ cc
pelo 		| ADP case
valor 		| NOUN conj
é 		| AUX cop
ok 		| PUNCT ROOT
. 		| PUNCT punct
Enteguega 		| PROPN ROOT
o 		| PRON det
que 		| PRON obj
propõe 		| VERB acl:relcl
. 		| PUNCT punct


In [13]:
displacy.render(doc, style="dep", options={"compact": "True"})

## Set of Syntactic Patterns

In [38]:
def extrair_padroes_sintaticos(doc):
    pat_19 = []
    pat_20 = []
    pat_21 = []
    pat_22 = []
    pat_23 = []

    for token in doc:
        # ADJ → NOUN
        if token.pos_ == "NOUN":
            adjs = [w for w in token.lefts
                    if w.dep_ == "amod" and w.pos_ == "ADJ"]
            if adjs:
                pat_19.append({"noun": token.text, "adj": [w.text for w in adjs]})

        # ADV → ADJ → Not(NOUN)
        if token.pos_ == "VERB":
            subj_adj = [w for w in token.lefts
                        if w.dep_ == "nsubj" and w.pos_ == "ADJ"]
            adv = [w for w in token.rights
                   if w.dep_ == "advmod" and w.pos_ == "ADV"]
            intens = [w for a in adv
                      for w in a.children if w.pos_ == "ADV"]
            if subj_adj and adv:
                pat_20.append({"adj": [w.text for w in subj_adj],
                       "adv": [w.text for w in adv],
                       "intensificador": [w.text for w in intens]})
                
        if token.pos_ == "ADJ" and token.dep_ == "amod":
            head = token.head
            # Busca outro ADJ no mesmo NOUN pai
            irmaos_adj = [w for w in head.children
                          if w.pos_ == "ADJ" and w != token]
            if irmaos_adj and head.pos_ != "NOUN":
                pat_21.append({"adj1": token.text,
                       "adj2": [w.text for w in irmaos_adj],
                       "head": head.text})
                
        if token.pos_ == "ADJ" and token.dep_ == "ROOT":
            subj_noun = [w for w in token.lefts
                         if w.dep_ == "nsubj" and w.pos_ == "NOUN"]
            intensif = [w for w in token.children
                        if w.dep_ == "advmod" and w.pos_ == "ADV"]
            if subj_noun:
                 pat_22.append({"noun_subj": [w.text for w in subj_noun],
                       "adj_pred": token.text,
                       "intensificador": [w.text for w in intensif]})
                
        if token.pos_ == "VERB" and token.dep_ == "ROOT":
            # Busca particípio ou gerúndio dependente
            formas = [w for w in token.rights
                      if w.pos_ in ("VERB", "ADJ")
                      and any(f in w.morph.get("VerbForm", [])
                              for f in ["Part", "Ger"])]
            adv_intens = [w for w in token.children
                          if w.dep_ == "advmod" and w.pos_ == "ADV"]
            adv_forma = [w for f in formas
                         for w in f.children
                         if w.dep_ == "advmod"]
            if formas and (adv_intens or adv_forma):
                pat_23.append({"verbo_aux": token.text,
                       "forma_verbal": [w.text for w in formas],
                       "adv_intensif": [w.text for w in adv_intens],
                       "adv_forma": [w.text for w in adv_forma]})
                
        
    return {
        "ADJ_NOUN" : len(pat_19),
        "ADV_ADJ_NOT(NOUN)" : len(pat_20),
        "ADJ_ADJ_NOT(NOUN)" : len(pat_21),
        "NOUN_ADJ_NOT(NOUN)" : len(pat_22),
        "ADV_VERB_PCP_GER_PS_IMPF" : len(pat_23) ,
    }


In [ ]:
l = "Vergonhoso cobrarem tão caro."

doc = nlp(l.lower())
print(doc)
print(extrair_padroes_sintaticos(doc))

vergonhoso cobrarem tão caro.
{'ADJ_NOUN': 0, 'ADV_ADJ_NOT(NOUN)': 0, 'ADJ_ADJ_NOT(NOUN)': 0, 'NOUN_ADJ_NOT(NOUN)': 0, 'ADV_VERB_PCP_GER_PS_IMPF': 0}


In [ ]:
for token in doc:
    print(f"{token.text:15} pos={token.pos_:6} dep={token.dep_:12} head={token.head.text}")

vergonhoso      pos=ADJ    dep=ROOT         head=vergonhoso
cobrarem        pos=VERB   dep=csubj        head=vergonhoso
tão             pos=ADV    dep=advmod       head=caro
caro            pos=ADJ    dep=xcomp        head=cobrarem
.               pos=PUNCT  dep=punct        head=vergonhoso


{'ADJ_NOUN': 1, 'ADV_ADJ_NOT(NOUN)': 0, 'ADJ_ADJ_NOT(NOUN)': 0, 'NOUN_ADJ_NOT(NOUN)': 0, 'ADV_VERB_PCP_GER_PS_IMPF': 0}
{'ADJ_NOUN': 1, 'ADV_ADJ_NOT(NOUN)': 0, 'ADJ_ADJ_NOT(NOUN)': 0, 'NOUN_ADJ_NOT(NOUN)': 0, 'ADV_VERB_PCP_GER_PS_IMPF': 0}
{'ADJ_NOUN': 0, 'ADV_ADJ_NOT(NOUN)': 0, 'ADJ_ADJ_NOT(NOUN)': 0, 'NOUN_ADJ_NOT(NOUN)': 0, 'ADV_VERB_PCP_GER_PS_IMPF': 0}
{'ADJ_NOUN': 0, 'ADV_ADJ_NOT(NOUN)': 0, 'ADJ_ADJ_NOT(NOUN)': 0, 'NOUN_ADJ_NOT(NOUN)': 0, 'ADV_VERB_PCP_GER_PS_IMPF': 0}
{'ADJ_NOUN': 0, 'ADV_ADJ_NOT(NOUN)': 0, 'ADJ_ADJ_NOT(NOUN)': 0, 'NOUN_ADJ_NOT(NOUN)': 0, 'ADV_VERB_PCP_GER_PS_IMPF': 0}
{'ADJ_NOUN': 0, 'ADV_ADJ_NOT(NOUN)': 0, 'ADJ_ADJ_NOT(NOUN)': 0, 'NOUN_ADJ_NOT(NOUN)': 0, 'ADV_VERB_PCP_GER_PS_IMPF': 0}
{'ADJ_NOUN': 0, 'ADV_ADJ_NOT(NOUN)': 0, 'ADJ_ADJ_NOT(NOUN)': 0, 'NOUN_ADJ_NOT(NOUN)': 0, 'ADV_VERB_PCP_GER_PS_IMPF': 0}
{'ADJ_NOUN': 0, 'ADV_ADJ_NOT(NOUN)': 0, 'ADJ_ADJ_NOT(NOUN)': 0, 'NOUN_ADJ_NOT(NOUN)': 0, 'ADV_VERB_PCP_GER_PS_IMPF': 0}
{'ADJ_NOUN': 0, 'ADV_ADJ_NOT(NOUN)': 0, 

KeyboardInterrupt: 